In [9]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.preprocessing import image
import os
import glob
from tqdm import tqdm  # Progress bar
from collections import defaultdict

# Load the trained EfficientNetB0 model
model = load_model("efficientnet_fruit_classifier.h5")

# Get the base EfficientNetB0 model
base_model = model # Ensure this layer name matches your model

# Define feature extractors from different depths of EfficientNetB0
early_feature_extractor = Model(inputs=base_model.input, outputs=base_model.get_layer("block1a_activation").output)
middle_feature_extractor = Model(inputs=base_model.input, outputs=base_model.get_layer("block3a_expand_activation").output)
final_feature_extractor = Model(inputs=base_model.input, outputs=base_model.get_layer("top_activation").output)

# Define test image path
test_images_path = "./dataset/test"

# Define class labels
class_labels = ["freshapples", "freshbanana", "freshoranges", "rottenapples", "rottenbanana", "rottenoranges"]

# Store selected image paths for each class
class_images = defaultdict(list)

# Collect images for each class
for class_name in class_labels:
    class_path = os.path.join(test_images_path, class_name)
    if os.path.exists(class_path):
        image_paths = glob.glob(os.path.join(class_path, "*.*"))  # Load image files only
        class_images[class_name] = image_paths[:100]  # Take first 100 images

# Extract and save features for each class
for class_name, img_list in class_images.items():
    all_features = []
    
    print(f" Extracting features for {class_name} ({len(img_list)} images)...")
    
    for img_path in tqdm(img_list, desc=f"Processing {class_name}"):
        try:
            # Load and preprocess image
            img = image.load_img(img_path, target_size=(224, 224))
            img_array = image.img_to_array(img)
            img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension
            img_array = img_array / 255.0  # Normalize

            # Extract features
            early_features = early_feature_extractor.predict(img_array).flatten()
            middle_features = middle_feature_extractor.predict(img_array).flatten()
            final_features = final_feature_extractor.predict(img_array).flatten()

            # Combine features
            combined_features = np.concatenate([early_features, middle_features, final_features])
            all_features.append(combined_features)
        
        except Exception as e:
            print(f"Error processing {img_path}: {e}")

    # Convert to numpy array
    all_features = np.array(all_features)

    # Save features to a separate file for each class
    feature_filename = f"efficienet_{class_name}_features_finale.npy"
    np.save(feature_filename, all_features)

    print(f" Features saved: {feature_filename} (Shape: {all_features.shape})")

print(" Feature extraction complete for all classes!")


 Extracting features for freshapples (100 images)...


Processing freshapples:   0%|          | 0/100 [00:00<?, ?it/s]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 475ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 646ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step


Processing freshapples:   1%|          | 1/100 [00:03<05:59,  3.63s/it]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step


Processing freshapples:   2%|▏         | 2/100 [00:04<02:51,  1.75s/it]

... (1,203 outputs left out)

Processing rottenoranges:  99%|█████████▉| 99/100 [00:43<00:00,  2.40it/s]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step


Processing rottenoranges: 100%|██████████| 100/100 [00:43<00:00,  2.29it/s]


 Features saved: efficienet_rottenoranges_features_finale.npy (Shape: (100, 915712))
 Feature extraction complete for all classes!


In [1]:
import os
import numpy as np

# Class labels and feature filenames
class_labels = ["freshapples", "freshbanana", "freshoranges", "rottenapples", "rottenbanana", "rottenoranges"]
prefix = "efficienet_"
suffix = "_features_finale.npy"

all_verified = True

for class_name in class_labels:
    filename = f"{prefix}{class_name}{suffix}"
    
    if not os.path.exists(filename):
        print(f"[❌] Feature file missing: {filename}")
        all_verified = False
        continue

    try:
        features = np.load(filename)
        if features.shape[0] != 100:
            print(f"[⚠️] {filename}: Expected 100 features, found {features.shape[0]}")
            all_verified = False
        else:
            print(f"[✅] {filename} loaded successfully with shape: {features.shape}")
    except Exception as e:
        print(f"[❌] Error loading {filename}: {e}")
        all_verified = False

if all_verified:
    print("\n🎉 All feature files verified successfully!")
else:
    print("\n⚠️ Some files are missing or incomplete. Please check the logs above.")


[✅] efficienet_freshapples_features_finale.npy loaded successfully with shape: (100, 915712)
[✅] efficienet_freshbanana_features_finale.npy loaded successfully with shape: (100, 915712)
[✅] efficienet_freshoranges_features_finale.npy loaded successfully with shape: (100, 915712)
[✅] efficienet_rottenapples_features_finale.npy loaded successfully with shape: (100, 915712)
[✅] efficienet_rottenbanana_features_finale.npy loaded successfully with shape: (100, 915712)
[✅] efficienet_rottenoranges_features_finale.npy loaded successfully with shape: (100, 915712)

🎉 All feature files verified successfully!


In [11]:
import numpy as np
from sklearn.decomposition import PCA
import os
from tqdm import tqdm

# Define class labels
class_labels = ["freshapples", "freshbanana", "freshoranges", "rottenapples", "rottenbanana", "rottenoranges"]

# Define number of components for PCA
pca_components = 100  # Adjust based on dimensionality and variance explained

# Perform PCA and save results
for class_name in class_labels:
    print(f"Processing PCA for {class_name}...")
    
    # Load features
    feature_filename = f"efficienet_{class_name}_features_finale.npy"
    if not os.path.exists(feature_filename):
        print(f"Skipping {class_name}, file not found: {feature_filename}")
        continue
    
    all_features = np.load(feature_filename)
    
    # Split features into early, middle, and final layers
    split_idx1 = all_features.shape[1] // 3
    split_idx2 = 2 * split_idx1
    
    early_features = all_features[:, :split_idx1]
    middle_features = all_features[:, split_idx1:split_idx2]
    final_features = all_features[:, split_idx2:]
    
    # Case 1: Apply PCA separately to each layer, then fuse them
    pca_early = PCA(n_components=pca_components)
    pca_middle = PCA(n_components=pca_components)
    pca_final = PCA(n_components=pca_components)
    
    early_pca = pca_early.fit_transform(early_features)
    middle_pca = pca_middle.fit_transform(middle_features)
    final_pca = pca_final.fit_transform(final_features)
    
    fused_pca = np.concatenate([early_pca, middle_pca, final_pca], axis=1)
    
    # Save fused PCA output
    fused_pca_filename = f"efficient_{class_name}_fused_pca_final.npy"
    np.save(fused_pca_filename, fused_pca)
    print(f"Saved fused PCA features: {fused_pca_filename} (Shape: {fused_pca.shape})")
    
    # Case 2: Apply PCA to the combined feature layer
    pca_combined = PCA(n_components=pca_components)
    combined_pca = pca_combined.fit_transform(all_features)
    
    # Save combined PCA output
    combined_pca_filename = f"efficient_{class_name}_combined_pca_final.npy"
    np.save(combined_pca_filename, combined_pca)
    print(f"Saved combined PCA features: {combined_pca_filename} (Shape: {combined_pca.shape})")

print("PCA processing complete for all classes!")

Processing PCA for freshapples...
Saved fused PCA features: efficient_freshapples_fused_pca_final.npy (Shape: (100, 300))
Saved combined PCA features: efficient_freshapples_combined_pca_final.npy (Shape: (100, 100))
Processing PCA for freshbanana...
Saved fused PCA features: efficient_freshbanana_fused_pca_final.npy (Shape: (100, 300))
Saved combined PCA features: efficient_freshbanana_combined_pca_final.npy (Shape: (100, 100))
Processing PCA for freshoranges...
Saved fused PCA features: efficient_freshoranges_fused_pca_final.npy (Shape: (100, 300))
Saved combined PCA features: efficient_freshoranges_combined_pca_final.npy (Shape: (100, 100))
Processing PCA for rottenapples...
Saved fused PCA features: efficient_rottenapples_fused_pca_final.npy (Shape: (100, 300))
Saved combined PCA features: efficient_rottenapples_combined_pca_final.npy (Shape: (100, 100))
Processing PCA for rottenbanana...
Saved fused PCA features: efficient_rottenbanana_fused_pca_final.npy (Shape: (100, 300))
Saved 